In [0]:
# DBTITLE 1,Configuration and imports
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType, LongType, DoubleType
from datetime import datetime, timezone
import json

# ── Configuration ──────────────────────────────────────────────────────
CATALOG          = "bfsi"
SCHEMA           = "bronze_layer"
AUDIT_LOG_TABLE  = f"{CATALOG}.{SCHEMA}.abc_audit_log"

# Thresholds (test environment – production: 400_000 / 600_000)
CARD_MIN_RECORDS = 5_000
CARD_MAX_RECORDS = 50_000

NEFT_MIN_RECORDS = 1_000
NEFT_MAX_RECORDS = 30_000

UPI_MIN_RECORDS  = 5_000
UPI_MAX_RECORDS  = 100_000

BALANCE_TOLERANCE = 0.01          # ± tolerance for floating-point comparison

# Run metadata
RUN_TS   = datetime.now(timezone.utc)
RUN_ID   = RUN_TS.strftime("ABC_%Y%m%d_%H%M%S")

# Accumulator for all check results
abc_results = []

print(f"ABC Run: {RUN_ID}  |  Timestamp: {RUN_TS.isoformat()}")
print(f"Card record threshold:     {CARD_MIN_RECORDS:,} – {CARD_MAX_RECORDS:,}  (test mode)")
print(f"NEFT/RTGS record threshold: {NEFT_MIN_RECORDS:,} – {NEFT_MAX_RECORDS:,}  (test mode)")
print(f"UPI record threshold:      {UPI_MIN_RECORDS:,} – {UPI_MAX_RECORDS:,}  (test mode)")

In [0]:
# DBTITLE 1,AUDIT check – card record count per batch
# ── AUDIT CHECK: Record count per batch (all sources) ───────────────
card_df = spark.table(f"{CATALOG}.{SCHEMA}.card_transactions")
neft_df = spark.table(f"{CATALOG}.{SCHEMA}.neft_rtgs_transactions")
upi_df  = spark.table(f"{CATALOG}.{SCHEMA}.upi_transactions")

# Define audit configs: (dataframe, check_name, min, max)
audit_sources = [
    (card_df, "card_record_count",     CARD_MIN_RECORDS, CARD_MAX_RECORDS),
    (neft_df, "neft_rtgs_record_count", NEFT_MIN_RECORDS, NEFT_MAX_RECORDS),
    (upi_df,  "upi_record_count",      UPI_MIN_RECORDS,  UPI_MAX_RECORDS),
]

for src_df, check_name, min_rec, max_rec in audit_sources:
    audit_agg = (
        src_df
        .groupBy("_batch_id", "_file_name", "_source_system")
        .agg(F.count("*").alias("record_count"))
        .withColumn(
            "audit_passed",
            F.col("record_count").between(min_rec, max_rec)
        )
    )

    for row in audit_agg.collect():
        passed = row["audit_passed"]
        detail = {
            "batch_id": row["_batch_id"],
            "file_name": row["_file_name"],
            "record_count": row["record_count"],
            "expected_range": f"{min_rec}-{max_rec}",
        }
        abc_results.append({
            "run_id": RUN_ID,
            "check_type": "AUDIT",
            "source_system": row["_source_system"],
            "check_name": check_name,
            "passed": passed,
            "detail": json.dumps(detail),
            "check_ts": RUN_TS,
        })
        status = "PASS" if passed else "FAIL"
        print(f"AUDIT | {check_name} | {row['_batch_id']} | count={row['record_count']:,} | {status}")

In [0]:
# DBTITLE 1,BALANCE check – NEFT/RTGS settlement reconciliation
# ── BALANCE CHECK: NEFT/RTGS settlement amount reconciliation ─────────
neft_df = spark.table(f"{CATALOG}.{SCHEMA}.neft_rtgs_transactions")

balance_check = (
    neft_df
    .groupBy("msg_id", "_file_name", "_batch_id", "_source_system", "_total_settlement_amt")
    .agg(
        F.sum("amount").alias("sum_txn_amounts"),
        F.count("*").alias("txn_count"),
    )
    .withColumn(
        "difference",
        F.abs(F.col("sum_txn_amounts") - F.col("_total_settlement_amt"))
    )
    .withColumn(
        "balance_passed",
        F.col("difference") <= BALANCE_TOLERANCE
    )
)

balance_rows = balance_check.collect()

for row in balance_rows:
    passed = row["balance_passed"]
    detail = {
        "msg_id": row["msg_id"],
        "batch_id": row["_batch_id"],
        "file_name": row["_file_name"],
        "sum_txn_amounts": row["sum_txn_amounts"],
        "header_settlement_amt": row["_total_settlement_amt"],
        "difference": row["difference"],
        "txn_count": row["txn_count"],
        "tolerance": BALANCE_TOLERANCE,
    }
    abc_results.append({
        "run_id": RUN_ID,
        "check_type": "BALANCE",
        "source_system": row["_source_system"],
        "check_name": "neft_rtgs_settlement_reconciliation",
        "passed": passed,
        "detail": json.dumps(detail),
        "check_ts": RUN_TS,
    })
    status = "PASS" if passed else "FAIL"
    print(
        f"BALANCE | {row['msg_id']} | SUM={row['sum_txn_amounts']:,.2f} "
        f"vs HDR={row['_total_settlement_amt']:,.2f} | diff={row['difference']:.4f} | {status}"
    )

In [0]:
# DBTITLE 1,CONTROL check – duplicate detection
# ── CONTROL CHECK: Duplicate detection ────────────────────────────

# 1. Intra-source duplicates: card_transactions
card_dupes = (
    card_df
    .groupBy("txn_id", "_source_system")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)
card_dupe_count = card_dupes.count()

# 2. Intra-source duplicates: neft_rtgs_transactions
neft_dupes = (
    neft_df
    .groupBy("instr_id", "_source_system")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)
neft_dupe_count = neft_dupes.count()

# 3. Intra-source duplicates: upi_transactions
upi_dupes = (
    upi_df
    .groupBy("upi_txn_id", "_source_system")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)
upi_dupe_count = upi_dupes.count()

# 4. Cross-source duplicates (all pairwise ID collisions)
card_ids = card_df.select(F.col("txn_id").alias("id"), F.lit("CARD").alias("src"))
neft_ids = neft_df.select(F.col("instr_id").alias("id"), F.lit("NEFT_RTGS").alias("src"))
upi_ids  = upi_df.select(F.col("upi_txn_id").alias("id"), F.lit("UPI").alias("src"))

all_ids = card_ids.unionByName(neft_ids).unionByName(upi_ids)

cross_dupes = (
    all_ids
    .groupBy("id")
    .agg(
        F.count("*").alias("cnt"),
        F.collect_set("src").alias("sources")
    )
    .filter((F.col("cnt") > 1) & (F.size("sources") > 1))
)
cross_dupe_count = cross_dupes.count()

for label, dupe_cnt, source, dupe_df, id_col in [
    ("card_intra_source_duplicates",      card_dupe_count, "CARD",      card_dupes, "txn_id"),
    ("neft_rtgs_intra_source_duplicates", neft_dupe_count, "NEFT_RTGS", neft_dupes, "instr_id"),
    ("upi_intra_source_duplicates",       upi_dupe_count,  "UPI",       upi_dupes,  "upi_txn_id"),
    ("cross_source_id_collision",         cross_dupe_count, "ALL",      cross_dupes, "id"),
]:
    passed = dupe_cnt == 0
    detail = {"duplicate_id_count": dupe_cnt}
    # Capture sample duplicate IDs for debugging (max 10)
    if not passed:
        samples = [r[id_col] for r in dupe_df.limit(10).collect()]
        detail["sample_ids"] = samples

    abc_results.append({
        "run_id": RUN_ID,
        "check_type": "CONTROL",
        "source_system": source,
        "check_name": label,
        "passed": passed,
        "detail": json.dumps(detail),
        "check_ts": RUN_TS,
    })
    status = "PASS" if passed else "FAIL"
    print(f"CONTROL | {label} | duplicates={dupe_cnt:,} | {status}")

In [0]:
# DBTITLE 1,Persist ABC results to audit log (7-year retention)
# ── Persist results to abc_audit_log (7-year regulatory retention) ─────
results_schema = StructType([
    StructField("run_id", StringType()),
    StructField("check_type", StringType()),
    StructField("source_system", StringType()),
    StructField("check_name", StringType()),
    StructField("passed", BooleanType()),
    StructField("detail", StringType()),
    StructField("check_ts", TimestampType()),
])

results_df = spark.createDataFrame(abc_results, schema=results_schema)

# Create table with 7-year retention if it doesn't exist
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_LOG_TABLE} (
        run_id          STRING,
        check_type      STRING,
        source_system   STRING,
        check_name      STRING,
        passed          BOOLEAN,
        detail          STRING,
        check_ts        TIMESTAMP
    )
    USING DELTA
    COMMENT 'ABC check audit log – regulatory 7-year retention'
    TBLPROPERTIES (
        'delta.deletedFileRetentionDuration' = 'interval 2555 days',
        'delta.logRetentionDuration'         = 'interval 2555 days'
    )
""")

results_df.write.mode("append").saveAsTable(AUDIT_LOG_TABLE)

print(f"\n Logged {len(abc_results)} check result(s) to {AUDIT_LOG_TABLE}")
display(results_df)

In [0]:
# DBTITLE 1,Evaluate results and signal Airflow BranchOperator
# ── Evaluate overall status & signal Airflow BranchOperator ────────────
all_passed = all(r["passed"] for r in abc_results)
failed_checks = [r for r in abc_results if not r["passed"]]

print("=" * 70)
if all_passed:
    print("ALL ABC CHECKS PASSED — Silver job may proceed.")
else:
    print("ABC CHECK FAILURE(S) DETECTED — Silver job will be halted.")
    for f in failed_checks:
        print(f"    ▸ [{f['check_type']}] {f['check_name']} | {f['source_system']}")
        print(f"      Detail: {f['detail']}")
print("=" * 70)

# Set task value for Airflow BranchOperator integration
# Airflow reads this via `databricks.sdk` or the Jobs API to decide branching
try:
    dbutils.jobs.taskValues.set(key="abc_check_passed", value=all_passed)
    dbutils.jobs.taskValues.set(key="abc_run_id",       value=RUN_ID)
    print(f"\n🔗 Task values set: abc_check_passed={all_passed}, abc_run_id={RUN_ID}")
except Exception:
    # Running interactively (not as a job task)
    print(f"\n Interactive mode — task values not set (abc_check_passed={all_passed})")

# Fail the notebook if any check failed (stops downstream Airflow tasks)
if not all_passed:
    raise Exception(
        f"ABC Check Failed — {len(failed_checks)} failure(s) detected. "
        f"Run ID: {RUN_ID}. See {AUDIT_LOG_TABLE} for details."
    )